# Configuração e teste do banco — ETL 2020
Este notebook **não cria tabelas e não carrega voos**. Ele testa os dados do `.env` e lista a situação do schema.

No pgAdmin, registre a conexão com o servidor e crie `vra_dw_2020`. Na raiz do projeto, configure o `.env` a partir de `.env.example`, mantendo `PGDATABASE=vra_dw_2020` e `PGSCHEMA=etl_2020`. Nunca envie o `.env` ao GitHub.

Instale as dependências do projeto antes de executar. No Colab, use ambiente local para alcançar o PostgreSQL do seu Mac. [Documentação oficial](https://research.google.com/colaboratory/local-runtimes.html).

In [ ]:
from pathlib import Path
import sys

raiz = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/etl_vra_anac_2020.py").is_file()), None)
if raiz is None:
    raise FileNotFoundError("Abra o notebook dentro do checkout atualizado do projeto.")
sys.path.insert(0, str(raiz / "scripts"))
import etl_vra_anac_2020 as etl
import pandas as pd
from IPython.display import display
print("Contrato:", etl.VERSAO, "| Pandas:", pd.__version__)

In [ ]:
from sqlalchemy import text
engine, schema = etl.conectar(raiz / ".env")
try:
    with engine.connect() as conn:
        conn.execute(text("SET TRANSACTION READ ONLY"))
        banco, usuario = conn.execute(text("SELECT current_database(), current_user")).one()
        tabelas = conn.execute(text("SELECT table_name FROM information_schema.tables WHERE table_schema=:s ORDER BY table_name"), {"s": schema}).scalars().all()
        print("Conexão aprovada:", banco, "| usuário:", usuario)
        print("Schema de destino:", schema, "| tabelas existentes:", tabelas)
finally:
    engine.dispose()

Conexão funcionando: volte ao notebook do ETL. A carga continua desligada até você ativar `EXECUTAR_CARGA`. O schema é criado somente nessa carga, não neste teste. Se já houver tabelas, confira a origem antes de prosseguir; a implementação recusa substituir outra execução.